In [1]:
import pandas as pd
import numpy as np

## Loading & Preparing

In [2]:
df_raw = pd.read_csv("../data/external/Ponds1.csv", low_memory=False)
df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 74796 entries, 0 to 74795
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   station          74796 non-null  str    
 1   Date             74796 non-null  str    
 2   Time             74745 non-null  str    
 3   NITRATE(PPM)     74794 non-null  str    
 4   PH               74795 non-null  str    
 5   AMMONIA(mg/l)    74796 non-null  float64
 6   TEMP             74790 non-null  float64
 7   DO               74790 non-null  str    
 8   TURBIDITY        74796 non-null  float64
 9   MANGANESE(mg/l)  74775 non-null  str    
dtypes: float64(3), str(7)
memory usage: 5.7 MB


In [3]:
df_raw.head()

,station,Date,Time,NITRATE(PPM),PH,AMMONIA(mg/l),TEMP,DO,TURBIDITY,MANGANESE(mg/l)
0,station1,01-02-2022,08:00:00,18.3,5.7,0.010,23.20,11.6,31.7,0.71
1,station1,01-02-2022,08:20:00,3.6,5.1,0.094,23.41,10.5,18.8,0.62
2,station1,01-02-2022,08:40:00,13.1,5.5,0.060,23.63,10.3,23.2,0.73
3,station1,01-02-2022,09:00:00,18.1,5.2,0.018,23.64,9.4,26.7,0.64
4,station1,01-02-2022,09:20:00,10.8,5.2,0.038,23.81,8.8,19.5,0.68


##### Data type Fixing

In [46]:

df_raw['station'] = df_raw['station'].str.lower().str.strip()

sensor_cols = ['DO','PH','AMMONIA(mg/l)','TEMP','NITRATE(PPM)','TURBIDITY', 'MANGANESE(mg/l)']
for col in sensor_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df_raw['datetime'] = pd.to_datetime(
    df_raw['Date'] + ' ' + df_raw['Time'], dayfirst=True, errors='coerce'
)

df_raw.dtypes

station                       str
Date                          str
Time                          str
NITRATE(PPM)              float64
PH                        float64
AMMONIA(mg/l)             float64
TEMP                      float64
DO                        float64
TURBIDITY                 float64
MANGANESE(mg/l)           float64
datetime           datetime64[us]
dtype: object

### Handling mising values

In [47]:
df = df_raw[df_raw['station'] == 'station2'].copy()
df = df.sort_values('datetime').reset_index(drop=True)
print(f"rows: {df.shape[0]:,} | features: {df.shape[1]}\n")

missing = df[sensor_cols + ["datetime"]].isna().sum()
print(missing[missing > 0])

rows: 25,498 | features: 11

DO                  6
PH                  1
NITRATE(PPM)        2
MANGANESE(mg/l)    21
datetime           17
dtype: int64


In [48]:
print("\nDecision: DROP rows with any NaN in features/datetime")

df.dropna(subset=sensor_cols + ["datetime"], inplace=True)
print(f"rows: {df.shape[0]:,} | features: {df.shape[1]}\n")

df.isna().sum()


Decision: DROP rows with any NaN in features/datetime
rows: 25,455 | features: 11



station            0
Date               0
Time               0
NITRATE(PPM)       0
PH                 0
AMMONIA(mg/l)      0
TEMP               0
DO                 0
TURBIDITY          0
MANGANESE(mg/l)    0
datetime           0
dtype: int64

In [49]:

# Drop isolated temperature sensor faults (where TEMP is exactly 0)
df = df[df['TEMP'] > 5].reset_index(drop=True)

# Drop isolated dissolved oxygen sensor faults (where DO is exactly 0)
df = df[df['DO'] != 0].reset_index(drop=True)

print(f"rows: {df.shape[0]:,} | features: {df.shape[1]}\n")

rows: 25,429 | features: 11



## Labeling Logics
   Each parameter is tested against a single published safe
   threshold. If ANY one parameter falls outside its safe
   range, the label is Halt Feeding (1).
   If ALL parameters are within safe ranges, label is Feed (0).

   This "worst parameter wins" rule reflects the biological
   reality that a fish cannot selectively ignore one stressor
   because the other parameters happen to be fine.


#### Dissolved Oxygen
 THRESHOLD: 5.0 mg/L  (below = Halt)

 SOURCE 1 — Boyd, C.E. (1990). Water Quality in Ponds for Aquaculture.
   Alabama Agricultural Experiment Station, Auburn University.
   Direct quote: "Dissolved oxygen concentrations below 5 mg/L cause
   fish to reduce their feeding activity. At concentrations below
   3 mg/L, most fish stop feeding entirely."
     We use 5.0 mg/L as the single binary threshold because Boyd
     identifies this as the point where feeding REDUCTION begins,
     meaning anything below is already sub-optimal for a fed pond.

 SOURCE 2 — Tran-Duy, A., Schrama, J.W., van Dam, A.A., Verreth, J.A.J.
   (2008). "Effects of oxygen concentration and body weight on the
   maximum feed intake, growth and haematological parameters of
   Nile tilapia, Oreochromis niloticus."
   Aquaculture, 275(1–4), 152–162.
   Direct quote: "Feed intake was significantly reduced at dissolved
   oxygen concentrations below 5 mg/L compared to saturation levels."
     Directly measured in tilapia. Catla is a comparable warm-water
     cyprinid with similar DO requirements (Das et al. 2012).

 SOURCE 3 — FAO (2014). Small-scale aquaponic food production.
   FAO Fisheries and Aquaculture Technical Paper No. 589. Rome.
   States 5 mg/L as the minimum recommended DO for fed aquaculture
   systems where fish are expected to consume full rations.

 MATH:<br>
   bad_DO = 1  if  DO < 5.0<br>
   bad_DO = 0  if  DO >= 5.0<br>

In [51]:
DO_THRESHOLD = 5.0  # mg/L

df['bad_DO'] = (df['DO'] < DO_THRESHOLD).astype(int)


print(f"  Rows flagged as unsafe: {df['bad_DO'].sum()} / {len(df)} "
      f"({df['bad_DO'].mean():.1%})")
print(f"  DO mean when safe  : {df[df['bad_DO']==0]['DO'].mean():.3f} mg/L")
print(f"  DO mean when unsafe: {df[df['bad_DO']==1]['DO'].mean():.3f} mg/L")

  Rows flagged as unsafe: 1983 / 25429 (7.8%)
  DO mean when safe  : 10.997 mg/L
  DO mean when unsafe: 3.353 mg/L


#### pH
 THRESHOLD: safe band = 6.5 to 8.5  (outside = Halt)

 SOURCE 1 — Boyd, C.E. (1990). Water Quality in Ponds for Aquaculture.
   Direct quote: "The optimum pH range for most warm-water fish is
   6.5 to 9.0, but the best growth occurs between 7 and 8. Values
   below 6.5 cause stress and reduced feeding in most species."
   → Boyd gives 6.5 as the lower safe limit. We use 8.5 as the
     upper limit (not 9.0) because:

 SOURCE 2 — Pillay, T.V.R. (2004). Aquaculture and the Environment.
   Fishing News Books, Oxford.
   States that for tilapia and catla specifically, the acceptable
   pH range for normal growth and feeding is 6.5–8.5. Above 8.5,
   ammonia toxicity increases sharply because more ammonia converts
   to the unionized NH3 form, compounding the stress.
   → The 8.5 upper limit is therefore justified both by direct pH
     stress AND by the interaction with ammonia toxicity.

 SOURCE 3 — Das, P., Mandal, S.C., Bhagabati, S.K., Akhtar, M.S.,
   Singh, S.K. (2012). "Important Aspects of Water Quality Parameters
   in Aquaculture: A Review." Proceedings of the National Academy
   of Sciences India Section B: Biological Sciences.
   Confirms 6.5–8.5 as the functional feeding range for catla
   (Catla catla) specifically.

 MATH:<br>
   bad_PH = 1  if  PH < 6.5  OR  PH > 8.5<br>
   bad_PH = 0  if  6.5 <= PH <= 8.5<br>

In [52]:
PH_LOW  = 6.5
PH_HIGH = 8.5

df['bad_PH'] = ((df['PH'] < PH_LOW) | (df['PH'] > PH_HIGH)).astype(int)


print(f"  Rows flagged as unsafe: {df['bad_PH'].sum()} / {len(df)} "
      f"({df['bad_PH'].mean():.1%})")
print(f"  pH mean when safe  : {df[df['bad_PH']==0]['PH'].mean():.3f}")
print(f"  pH mean when unsafe: {df[df['bad_PH']==1]['PH'].mean():.3f}")
print(f"  Breakdown — below 6.5 : {(df['PH'] < PH_LOW).sum()} rows")
print(f"  Breakdown — above 8.5 : {(df['PH'] > PH_HIGH).sum()} rows")

  Rows flagged as unsafe: 14171 / 25429 (55.7%)
  pH mean when safe  : 7.438
  pH mean when unsafe: 6.358
  Breakdown — below 6.5 : 11539 rows
  Breakdown — above 8.5 : 2632 rows


#### Temperature
 THRESHOLD: safe band = 22°C to 32°C  (outside = Halt)

 SOURCE 1 — Das et al. (2012) — same paper as above.
   Direct quote on catla specifically: "Catla catla grows optimally
   at water temperatures between 25°C and 32°C. Below 22°C,
   feeding and metabolic activity decline significantly. Above 35°C,
   thermal stress becomes severe."
   → Gives us 22°C as the lower binary cut. We use 32°C (not 35°C)
     as the upper cut, explained below.

 SOURCE 2 — Pillay, T.V.R. (2004). Aquaculture and the Environment.
   For tilapia: "Water temperatures above 32°C cause growth
   depression and reduced feed conversion efficiency, even when
   dissolved oxygen is maintained at adequate levels."
   → 32°C is the conservative upper limit used for both species
     since tilapia FCR degrades above this point.

 SOURCE 3 — Boyd, C.E. (1990).
   "Warm-water fish generally cease feeding when water temperature
   falls below 20°C and show markedly reduced feeding below 22°C."
   → Confirms 22°C as the lower threshold. Our data minimum for
     Station2 after removing sensor failures is 18.0°C, meaning
     the cold-night readings DO fall into the unsafe zone.

 MATH:<br>
   bad_TEMP = 1  if  TEMP < 22.0  OR  TEMP > 32.0<br>
   bad_TEMP = 0  if  22.0 <= TEMP <= 32.0

In [53]:
TEMP_LOW  = 22.0   # °C
TEMP_HIGH = 32.0   # °C

df['bad_TEMP'] = ((df['TEMP'] < TEMP_LOW) | (df['TEMP'] > TEMP_HIGH)).astype(int)



print(f"  Rows flagged as unsafe: {df['bad_TEMP'].sum()} / {len(df)} "
      f"({df['bad_TEMP'].mean():.1%})")
print(f"  TEMP mean when safe  : {df[df['bad_TEMP']==0]['TEMP'].mean():.3f} °C")
print(f"  TEMP mean when unsafe: {df[df['bad_TEMP']==1]['TEMP'].mean():.3f} °C")
print(f"  Breakdown — below 22°C: {(df['TEMP'] < TEMP_LOW).sum()} rows")
print(f"  Breakdown — above 32°C: {(df['TEMP'] > TEMP_HIGH).sum()} rows")

  Rows flagged as unsafe: 7281 / 25429 (28.6%)
  TEMP mean when safe  : 26.845 °C
  TEMP mean when unsafe: 23.254 °C
  Breakdown — below 22°C: 5619 rows
  Breakdown — above 32°C: 1662 rows


#### AMMONIA
 THRESHOLD: 0.25 mg/L  (above = Halt)

 SOURCE 1 — Hargreaves, J.A. and Tucker, C.S. (2004).
   "Managing Ammonia in Fish Ponds."
   Southern Regional Aquaculture Center (SRAC) Publication No. 4603.
   Direct quote: "Total ammonia nitrogen concentrations above
   0.25 mg/L can cause sublethal stress in warm-water fish,
   including reduced feed intake and impaired immune function."
   → This is a SRAC publication — the primary applied aquaculture
     research body in the USA, widely cited for tropical species work.

 SOURCE 2 — Boyd, C.E. (1990).
   "Concentrations of un-ionized ammonia above 0.02 mg/L are
   harmful to fish, but total ammonia values below 0.25 mg/L
   are generally safe at the pH values found in most fish ponds."
   → Boyd links the 0.25 mg/L total ammonia threshold to typical
     pond pH conditions (6–8), which matches our dataset.

 NOTE ON IONIZATION:
   Total ammonia exists as NH4+ (safe, ionized) and NH3 (toxic,
   un-ionized). The toxic fraction depends on pH. At our pond's
   typical pH of 6.5–7.5, >99% is in the safe NH4+ form.
   This means the 0.25 mg/L total ammonia threshold is conservative
   and appropriate — at higher pH the threshold would be lower,
   but our pond conditions support this number directly.

 MATH:<br>
   bad_NH3 = 1  if  AMMONIA > 0.25<br>
   bad_NH3 = 0  if  AMMONIA <= 0.25

In [54]:
NH3_THRESHOLD = 0.25  # mg/L total ammonia

df['bad_NH3'] = (df['AMMONIA(mg/l)'] > NH3_THRESHOLD).astype(int)


print(f"  Rows flagged as unsafe: {df['bad_NH3'].sum()} / {len(df)} "
      f"({df['bad_NH3'].mean():.1%})")
print(f"  Ammonia mean when safe  : {df[df['bad_NH3']==0]['AMMONIA(mg/l)'].mean():.4f} mg/L")
print(f"  Ammonia mean when unsafe: {df[df['bad_NH3']==1]['AMMONIA(mg/l)'].mean():.4f} mg/L")

  Rows flagged as unsafe: 1225 / 25429 (4.8%)
  Ammonia mean when safe  : 0.0456 mg/L
  Ammonia mean when unsafe: 0.3781 mg/L


#### NITRATE
 THRESHOLD: 100 ppm  (above = Halt)

 SOURCE 1 — Boyd, C.E. (1990). Water Quality in Ponds for Aquaculture.
   Direct quote: "Nitrate is relatively non-toxic to fish at
   concentrations normally found in ponds. However, concentrations
   above 100 mg/L (ppm) may cause chronic stress and reduced
   feeding in sensitive species."
   → Boyd draws the line at 100 ppm for chronic stress effects.

 SOURCE 2 — FAO Fisheries Technical Paper (2014).
   Recommends nitrate below 100 ppm for fed pond systems to avoid
   long-term growth suppression and immune depression.

 DATA CONTEXT:
   Station2 nitrate ranges from 0 to 143 ppm.
   Only 505 rows (2.0%) exceed 100 ppm — so this threshold
   contributes rarely but captures genuine outlier events.

 MATH:<br>
   bad_NO3 = 1  if  NITRATE > 100.0<br>
   bad_NO3 = 0  if  NITRATE <= 100.0


In [55]:
NO3_THRESHOLD = 100.0  # ppm

df['bad_NO3'] = (df['NITRATE(PPM)'] > NO3_THRESHOLD).astype(int)


print(f"  Rows flagged as unsafe: {df['bad_NO3'].sum()} / {len(df)} "
      f"({df['bad_NO3'].mean():.1%})")
print(f"  Nitrate mean when safe  : {df[df['bad_NO3']==0]['NITRATE(PPM)'].mean():.3f} ppm")
print(f"  Nitrate mean when unsafe: {df[df['bad_NO3']==1]['NITRATE(PPM)'].mean():.3f} ppm")

  Rows flagged as unsafe: 494 / 25429 (1.9%)
  Nitrate mean when safe  : 22.269 ppm
  Nitrate mean when unsafe: 121.954 ppm


#### TURBIDITY
 THRESHOLD: 40 NTU  (above = Halt)

 SOURCE 1 — Boyd, C.E. (1990). Water Quality in Ponds for Aquaculture.
   Direct quote: "Turbidity in excess of 40 NTU from suspended
   inorganic particles reduces light penetration, inhibits
   phytoplankton photosynthesis, and interferes with the feeding
   behaviour of sight-feeding fish."
   → Boyd gives 40 NTU as the operational threshold above which
     feeding behaviour is measurably disrupted.

 SOURCE 2 — FAO (2014).
   Recommends turbidity below 40 NTU for managed fed ponds.
   Above this, reduced photosynthesis leads to DO depression
   during overnight hours, creating a compound risk.



 MATH:<br>
   bad_TURB = 1  if  TURBIDITY > 40.0<br>
   bad_TURB = 0  if  TURBIDITY <= 40.0

In [56]:
TURB_THRESHOLD = 40.0  # NTU

df['bad_TURB'] = (df['TURBIDITY'] > TURB_THRESHOLD).astype(int)


print(f"  Rows flagged as unsafe: {df['bad_TURB'].sum()} / {len(df)} "
      f"({df['bad_TURB'].mean():.1%})")
print(f"  Turbidity mean when safe  : {df[df['bad_TURB']==0]['TURBIDITY'].mean():.3f} NTU")
print(f"  Turbidity mean when unsafe: {df[df['bad_TURB']==1]['TURBIDITY'].mean():.3f} NTU")

  Rows flagged as unsafe: 528 / 25429 (2.1%)
  Turbidity mean when safe  : 27.459 NTU
  Turbidity mean when unsafe: 42.512 NTU



   Station2 turbidity is entirely within 10–46 NTU.
   These are well-managed IoT-monitored research ponds.
   Only 528 rows (2.1%) exceed 40 NTU — genuine spike events.

In [57]:
df.head()

,station,Date,Time,NITRATE(PPM),PH,AMMONIA(mg/l),TEMP,DO,TURBIDITY,MANGANESE(mg/l),datetime,bad_DO,bad_PH,bad_TEMP,bad_NH3,bad_NO3,bad_TURB
0,station2,01-02-2022,00:00:00,29.172,5.656,0.022088,22.91168,8.21866,29.9592,0.64480,2022-02-01 00:00:00,0,1,0,0,0,0
1,station2,01-02-2022,00:20:00,14.280,5.858,0.071284,22.77000,6.23826,32.8320,0.65472,2022-02-01 00:20:00,0,1,0,0,0,0
2,station2,01-02-2022,00:40:00,5.712,5.454,0.058232,22.73964,6.33728,25.6500,0.65472,2022-02-01 00:40:00,0,1,0,0,0,0
3,station2,01-02-2022,01:00:00,21.114,5.353,0.018072,22.33484,10.00102,21.1356,0.71424,2022-02-01 01:00:00,0,1,0,0,0,0
4,station2,01-02-2022,01:20:00,7.446,5.050,0.060240,22.14256,11.09024,24.2136,0.66464,2022-02-01 01:20:00,0,1,0,0,0,0


## Final Label Assembly
 RULE: feed_label = max(bad_DO, bad_PH, bad_TEMP, bad_NH3, bad_NO3, bad_TURB)

  - If ALL six flags are 0 → Feed (everything is within safe range)
  - If ANY one flag is 1   → Halt (at least one parameter is unsafe)

 WHY MAX AND NOT SUM?<br>
   Using sum would mean a mild DO problem + mild pH problem = worse
   than either alone, which is biologically true BUT would require
   a weighted scoring system to be fair. Using max keeps the logic
   simple, transparent, and directly tied to individual thresholds
   that each have their own published source. Every Halt decision
   can be traced back to exactly which parameter caused it.


In [58]:
bad_cols = ['bad_DO','bad_PH','bad_TEMP','bad_NH3','bad_NO3','bad_TURB']
df['feed_label'] = df[bad_cols].max(axis=1).astype(int)

In [40]:
disp = ['DO','PH','TEMP','AMMONIA(mg/l)','NITRATE(PPM)','TURBIDITY']
print("\nClass Separation Check:")
for col in disp:
    feed_mean = df[df['feed_label'] == 0][col].mean()
    halt_mean = df[df['feed_label'] == 1][col].mean()
    
    if col == 'DO':
        valid = feed_mean > halt_mean
    elif col == 'PH':
        valid = abs(feed_mean - 7.5) < abs(halt_mean - 7.5)
    elif col == 'TEMP':
        valid = abs(feed_mean - 27.0) < abs(halt_mean - 27.0)
    else:
        valid = feed_mean < halt_mean
        
    status = "PASS" if valid else "FAIL"
    print(f"{col:15} | Feed: {feed_mean:8.3f} | Halt: {halt_mean:8.3f} | Status: {status}")


Class Separation Check:
DO              | Feed:   12.308 | Halt:    9.707 | Status: PASS
PH              | Feed:    7.480 | Halt:    6.603 | Status: PASS
TEMP            | Feed:   26.484 | Halt:   25.572 | Status: PASS
AMMONIA(mg/l)   | Feed:    0.031 | Halt:    0.073 | Status: PASS
NITRATE(PPM)    | Feed:   21.172 | Halt:   25.305 | Status: PASS
TURBIDITY       | Feed:   27.628 | Halt:   27.827 | Status: PASS


In [59]:
stats = df.groupby('feed_label')['MANGANESE(mg/l)'].describe().round(3)
print(stats.to_string())

              count   mean    std  min    25%    50%    75%    max
feed_label                                                        
0            6784.0  0.599  0.231  0.2  0.397  0.596  0.801  1.000
1           18645.0  0.675  0.385  0.2  0.494  0.651  0.746  3.472


Manganese has high overlap and zero distinct variance between classes. Combined with the domain knowledge that Ghanaian smallholders cannot easily measure Manganese with standard manual kits, this statistical profile gives us data-driven reason to drop it.

In [41]:
# Select only the raw telemetry inputs, time backbone, and the target label
keep_cols = [
    'datetime', 
    'DO', 
    'PH', 
    'AMMONIA(mg/l)', 
    'TEMP', 
    'NITRATE(PPM)', 
    'TURBIDITY', 
    'feed_label'
]

# Create the clean dataframe
engineering_ready_df = df[keep_cols]

print("Columns retained for feature engineering:")
print(engineering_ready_df.columns.tolist())
print(f"Dataset shape: {engineering_ready_df.shape}")

Columns retained for feature engineering:
['datetime', 'DO', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'NITRATE(PPM)', 'TURBIDITY', 'feed_label']
Dataset shape: (25448, 8)


In [44]:
print("feed_label distribution:")
for cls, name in {0: "Feed", 1: "Halt Feeding"}.items():
    n = (engineering_ready_df["feed_label"] == cls).sum()
    print(f"  Class {cls} ({name}): {n:,}: {n/engineering_ready_df.shape[0]:.1%}")

feed_label distribution:
  Class 0 (Feed): 6,790: 26.7%
  Class 1 (Halt Feeding): 18,658: 73.3%


In [61]:
engineering_ready_df.head()

,datetime,DO,PH,AMMONIA(mg/l),TEMP,NITRATE(PPM),TURBIDITY,feed_label
0,2022-02-01 00:00:00,8.21866,5.656,0.022088,22.91168,29.172,29.9592,1
1,2022-02-01 00:20:00,6.23826,5.858,0.071284,22.77000,14.280,32.8320,1
2,2022-02-01 00:40:00,6.33728,5.454,0.058232,22.73964,5.712,25.6500,1
3,2022-02-01 01:00:00,10.00102,5.353,0.018072,22.33484,21.114,21.1356,1
4,2022-02-01 01:20:00,11.09024,5.050,0.060240,22.14256,7.446,24.2136,1


## Saving Final Data

In [60]:
engineering_ready_df.to_csv('../data/processed/station2_binary_labelled.csv', index=False)